In [ ]:
import xgboost as xgb
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score
import numpy as np

# Load data
benign_train = np.load('../data/tool_dataset/beavertails/reps/llama2-7b-chat-hf/30k/layers/layer_-15/benign.npy')
harmful_train = np.load('../data/tool_dataset/beavertails/reps/llama2-7b-chat-hf/30k/layers/layer_-15/harmful.npy')
benign_test = np.load('../data/tool_dataset/beavertails/reps/llama2-7b-chat-hf/30k/layers/layer_-15/test/benign.npy')
harmful_test = np.load('../data/tool_dataset/beavertails/reps/llama2-7b-chat-hf/30k/layers/layer_-15/test/harmful.npy')

# Create labels (0 for benign, 1 for harmful)
y_train = np.concatenate([
    np.zeros(len(benign_train)),
    np.ones(len(harmful_train))
])
y_test = np.concatenate([
    np.zeros(len(benign_test)),
    np.ones(len(harmful_test))
])

# Stack features
X_train = np.vstack([benign_train, harmful_train])
X_test = np.vstack([benign_test, harmful_test])

# Shuffle training data
shuffle_idx = np.random.permutation(len(X_train))
X_train = X_train[shuffle_idx]
y_train = y_train[shuffle_idx]

print(f"Training set: {X_train.shape}, Test set: {X_test.shape}")
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import xgboost as xgb
from sklearn.metrics import classification_report, confusion_matrix

# XGBoost with PCA preprocessing
pipeline_xgb = Pipeline([
    ('scaler', StandardScaler()),
    ('pca', PCA(n_components=0.95)),  # Same as your logistic regression
    ('xgb', xgb.XGBClassifier(
        objective='binary:logistic',
        max_depth=3,              # Shallower trees
        learning_rate=0.05,       # Lower learning rate
        n_estimators=200,         # More trees
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        n_jobs=-1
    ))
])

print("Training XGBoost with PCA...")
pipeline_xgb.fit(X_train, y_train)

y_pred_xgb = pipeline_xgb.predict(X_test)
print("\nXGBoost + PCA Results:")
print(classification_report(y_test, y_pred_xgb, target_names=['Benign', 'Harmful']))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_xgb))

# Compare with your logistic regression
print("\n" + "="*50)
print("Comparison:")
print(f"XGBoost + PCA accuracy: {(y_pred_xgb == y_test).mean():.4f}")

Training set: (27186, 4096), Test set: (3021, 4096)
Training XGBoost with PCA...

XGBoost + PCA Results:
              precision    recall  f1-score   support

      Benign       0.78      0.79      0.78      1288
     Harmful       0.84      0.83      0.84      1733

    accuracy                           0.81      3021
   macro avg       0.81      0.81      0.81      3021
weighted avg       0.81      0.81      0.81      3021


Confusion Matrix:
[[1014  274]
 [ 291 1442]]

Comparison:
XGBoost + PCA accuracy: 0.8130
